# AutoImmune Insight — Notebook 1: Data Cleaning & Preparation
**Google Data Analytics Capstone | Tayma Nofal**

This notebook covers:
- Loading raw datasets
- Handling missing values
- Standardizing disease codes (ICD-10)
- Filtering for autoimmune conditions
- Exporting clean data for analysis

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## Step 1: Load Datasets

We use two primary datasets:
- **NHANES** (CDC): Demographics, lab values, chronic conditions
- **Patient Survey Data** (Kaggle): Self-reported symptoms, diagnosis timelines, quality of life

Download links:
- NHANES: https://wwwn.cdc.gov/nchs/nhanes/
- Kaggle Autoimmune Dataset: https://www.kaggle.com/datasets (search: autoimmune disease patient survey)

In [ ]:
# --- Simulate dataset for demonstration purposes ---
# In your project, replace this with: pd.read_csv('../data/raw/nhanes.csv')

np.random.seed(42)
n = 500

diseases = ['Lupus', 'Rheumatoid Arthritis', 'Multiple Sclerosis',
            "Hashimoto's Thyroiditis", "Crohn's Disease",
            'Celiac Disease', 'Type 1 Diabetes', 'Psoriasis']

df = pd.DataFrame({
    'patient_id': range(1, n+1),
    'age': np.random.randint(18, 70, n),
    'gender': np.random.choice(['Female', 'Male', 'Non-binary'], n, p=[0.72, 0.26, 0.02]),
    'ethnicity': np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Other'], n, p=[0.55, 0.18, 0.15, 0.08, 0.04]),
    'disease': np.random.choice(diseases, n),
    'symptom_onset_age': np.random.randint(15, 60, n),
    'diagnosis_age': np.random.randint(20, 65, n),
    'fatigue': np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'joint_pain': np.random.choice([0, 1], n, p=[0.4, 0.6]),
    'brain_fog': np.random.choice([0, 1], n, p=[0.45, 0.55]),
    'skin_changes': np.random.choice([0, 1], n, p=[0.55, 0.45]),
    'gi_issues': np.random.choice([0, 1], n, p=[0.5, 0.5]),
    'has_insurance': np.random.choice([0, 1], n, p=[0.2, 0.8]),
    'saw_specialist': np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'qol_score': np.random.randint(20, 100, n),  # Quality of life 0-100
    'anti_inflammatory_diet': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    'regular_exercise': np.random.choice([0, 1], n, p=[0.5, 0.5]),
    'sleep_quality': np.random.choice(['Poor', 'Fair', 'Good'], n, p=[0.35, 0.35, 0.30]),
    'stress_management': np.random.choice([0, 1], n, p=[0.55, 0.45]),
})

# Ensure diagnosis is always after symptom onset
df['diagnosis_age'] = df.apply(
    lambda r: max(r['diagnosis_age'], r['symptom_onset_age'] + np.random.randint(1, 8)), axis=1
)

print(f'Dataset shape: {df.shape}')
df.head()

## Step 2: Data Quality Check

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Basic Stats ===')
df.describe()

## Step 3: Feature Engineering
Calculate **diagnosis delay** — the key metric for this analysis.

In [ ]:
# Diagnosis delay in years
df['diagnosis_delay_years'] = df['diagnosis_age'] - df['symptom_onset_age']

# Symptom burden score (count of reported symptoms)
symptom_cols = ['fatigue', 'joint_pain', 'brain_fog', 'skin_changes', 'gi_issues']
df['symptom_burden'] = df[symptom_cols].sum(axis=1)

# Lifestyle score (composite)
df['lifestyle_score'] = (
    df['anti_inflammatory_diet'] +
    df['regular_exercise'] +
    df['stress_management'] +
    (df['sleep_quality'] == 'Good').astype(int)
)

print('New features created:')
print(df[['patient_id', 'diagnosis_delay_years', 'symptom_burden', 'lifestyle_score']].head(10))

## Step 4: Export Clean Dataset

In [ ]:
df.to_csv('../data/cleaned/autoimmune_clean.csv', index=False)
print(f'Clean dataset saved: {df.shape[0]} rows, {df.shape[1]} columns')